In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

# 1. Define a Simple Backbone (Perfect for Ryzen 5)
class Conv4(nn.Module):
    def __init__(self):
        super(Conv4, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )

    def forward(self, x):
        x = self.layer(x)
        return x.view(x.size(0), -1) # Flatten to feature vector

# 2. Prototypical Loss Function (The Math Core)
def euclidean_dist(x, y):
    # x: [N_query, feature_dim], y: [N_way, feature_dim]
    n = x.size(0)
    m = y.size(0)
    d = x.size(1)
    x = x.unsqueeze(1).expand(n, m, d)
    y = y.unsqueeze(0).expand(n, m, d)
    return torch.pow(x - y, 2).sum(2)

# 3. Training Logic
def train_step(model, optimizer, support_images, query_images, n_way, k_shot):
    optimizer.zero_grad()
    
    # Embed everything
    support_embeddings = model(support_images) # [n_way * k_shot, dim]
    query_embeddings = model(query_images)     # [n_way * n_query, dim]
    
    # Calculate Prototypes (Average of support shots)
    prototypes = support_embeddings.view(n_way, k_shot, -1).mean(1)
    
    # Calculate Distances & Loss
    dists = euclidean_dist(query_embeddings, prototypes)
    log_p_y = torch.log_softmax(-dists, dim=1)
    
    # Target labels (0, 1, 2... for each query)
    target_labels = torch.arange(n_way).repeat_interleave(5).to(support_images.device)
    
    loss = nn.NLLLoss()(log_p_y, target_labels)
    loss.backward()
    optimizer.step()
    return loss.item()

# 4. Initialization
device = torch.device("cpu") # Use CPU for your Ryzen 5
model = Conv4().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model ready for training on Ryzen 5.")

Model ready for training on Ryzen 5.


In [18]:
import os
from PIL import Image
import random

class FewShotDataset:
    def __init__(self, root_dir, n_way, k_shot, k_query):
        self.root_dir = root_dir
        self.n_way = n_way
        self.k_shot = k_shot
        self.k_query = k_query
        
        # Get all subfolders (classes)
        self.classes = []
        for root, dirs, files in os.walk(root_dir):
            if len(files) > 0 and (files[0].endswith('.png') or files[0].endswith('.jpg')):
                self.classes.append(root)
        
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
        ])

    def get_episode(self):
        # 1. Select random classes
        selected_classes = random.sample(self.classes, self.n_way)
        support_images = []
        query_images = []

        for cls in selected_classes:
            all_imgs = [os.path.join(cls, f) for f in os.listdir(cls) if f.endswith(('.png', '.jpg'))]
            sampled = random.sample(all_imgs, self.k_shot + self.k_query)
            
            # Split into support and query
            for i, img_path in enumerate(sampled):
                img = self.transform(Image.open(img_path))
                if i < self.k_shot:
                    support_images.append(img)
                else:
                    query_images.append(img)

        return torch.stack(support_images).to(device), torch.stack(query_images).to(device)

In [ ]:
# Configuration
N_WAY = 5    # Number of classes per episode
K_SHOT = 5   # Number of images per class for support
K_QUERY = 5  # Number of images per class for testing
EPISODES = 1000 # Total training steps

# Initialize Dataset (Point this to your folder)
dataset = FewShotDataset(root_dir='images_background', n_way=N_WAY, k_shot=K_SHOT, k_query=K_QUERY)

print("Starting training...")

for ep in range(EPISODES):
    # Get a batch of images
    support_imgs, query_imgs = dataset.get_episode()
    
    # Run the training step you defined earlier
    loss = train_step(model, optimizer, support_imgs, query_imgs, N_WAY, K_SHOT)
    
    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1}/{EPISODES} - Loss: {loss:.4f}")

print("Training Complete!")

Starting training...
Episode 50/1000 - Loss: 0.2573
Episode 100/1000 - Loss: 0.0750
Episode 150/1000 - Loss: 0.1269
Episode 200/1000 - Loss: 0.3295


In [ ]:
torch.save(model.state_dict(), 'few_shot_model.pth')
print("Model saved as few_shot_model.pth")


Model saved as few_shot_model.pth


In [ ]:
def get_accuracy(log_p_y, target_labels):
    # Find the index of the highest probability
    _, y_hat = log_p_y.max(1)
    
    # Compare to actual labels and calculate mean
    acc = (y_hat == target_labels).float().mean()
    return acc.item()

# Updated Training Logic with Accuracy
def train_step_with_acc(model, optimizer, support_images, query_images, n_way, k_shot):
    optimizer.zero_grad()
    
    support_embeddings = model(support_images)
    query_embeddings = model(query_images)
    
    prototypes = support_embeddings.view(n_way, k_shot, -1).mean(1)
    dists = euclidean_dist(query_embeddings, prototypes)
    
    # We use -dists because the smallest distance should have the highest log-probability
    log_p_y = torch.log_softmax(-dists, dim=1)
    
    # Labels (assuming 5 queries per class like before)
    target_labels = torch.arange(n_way).repeat_interleave(5).to(support_images.device)
    
    loss = nn.NLLLoss()(log_p_y, target_labels)
    acc = get_accuracy(log_p_y, target_labels) # <--- Get Accuracy here
    
    loss.backward()
    optimizer.step()
    
    return loss.item(), acc

In [ ]:
for ep in range(EPISODES):
    support_imgs, query_imgs = dataset.get_episode()
    
    # Use the new function that returns both loss and accuracy
    loss, acc = train_step_with_acc(model, optimizer, support_imgs, query_imgs, N_WAY, K_SHOT)
    
    if (ep + 1) % 50 == 0:
        print(f"Episode {ep+1}/{EPISODES} | Loss: {loss:.4f} | Acc: {acc * 100:.2f}%")

Episode 50/1000 | Loss: 0.0023 | Acc: 100.00%
Episode 100/1000 | Loss: 0.1522 | Acc: 96.00%
Episode 150/1000 | Loss: 0.0093 | Acc: 100.00%
Episode 200/1000 | Loss: 0.0059 | Acc: 100.00%
Episode 250/1000 | Loss: 0.0032 | Acc: 100.00%
Episode 300/1000 | Loss: 0.0910 | Acc: 96.00%
Episode 350/1000 | Loss: 0.0094 | Acc: 100.00%
Episode 400/1000 | Loss: 0.0678 | Acc: 96.00%
Episode 450/1000 | Loss: 0.2372 | Acc: 84.00%
Episode 500/1000 | Loss: 0.0004 | Acc: 100.00%
Episode 550/1000 | Loss: 0.0096 | Acc: 100.00%
Episode 600/1000 | Loss: 0.0966 | Acc: 96.00%
Episode 650/1000 | Loss: 0.0061 | Acc: 100.00%
Episode 700/1000 | Loss: 0.0001 | Acc: 100.00%
Episode 750/1000 | Loss: 0.0165 | Acc: 100.00%
Episode 800/1000 | Loss: 0.0006 | Acc: 100.00%
Episode 850/1000 | Loss: 0.0006 | Acc: 100.00%
Episode 900/1000 | Loss: 0.0000 | Acc: 100.00%
Episode 950/1000 | Loss: 0.0010 | Acc: 100.00%
Episode 1000/1000 | Loss: 0.0288 | Acc: 100.00%


In [ ]:
# Save the trained model weights
torch.save(model.state_dict(), 'trained_fewshot_model.pth')
print("Training complete and model saved!")

Training complete and model saved!


In [ ]:
import os
print("Current Folder:", os.getcwd())
print("Folders here:", [d for d in os.listdir() if os.path.isdir(d)])

Current Folder: d:\python\majorassigment
Folders here: ['.gradio', 'images_background', 'images_background_small1', 'images_background_small2', 'images_evaluation']


In [ ]:
import os
import torch
from PIL import Image, ImageOps
from torchvision import transforms

# 1. AUTO-DETECT DATA DIRECTORY
data_dir = 'images_background'
all_prototypes = []
class_names = []

model.eval()
with torch.no_grad():
    # Loop through Alphabet folders (e.g., Sanskrit, Latin)
    for alphabet in sorted(os.listdir(data_dir)):
        alphabet_path = os.path.join(data_dir, alphabet)
        if not os.path.isdir(alphabet_path): continue
            
        # Loop through Character folders (e.g., character01)
        for char_folder in sorted(os.listdir(alphabet_path)):
            char_path = os.path.join(alphabet_path, char_folder)
            if not os.path.isdir(char_path): continue
                
            # Find the first image in the character folder
            images = [i for i in os.listdir(char_path) if i.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if images:
                img_path = os.path.join(char_path, images[0])
                class_names.append(f"{alphabet}_{char_folder}")
                
                # Process image for the prototype
                img = Image.open(img_path).convert('L')
                img = ImageOps.invert(img).resize((28, 28))
                t = transforms.ToTensor()(img).unsqueeze(0).to(device)
                all_prototypes.append(model(t))

# This will no longer be empty!
if len(all_prototypes) > 0:
    prototypes = torch.cat(all_prototypes)
    print(f"✅ Success! Created {len(class_names)} prototypes for your viva.")
else:
    print("❌ Critical: No images found. Check if images_background contains alphabet folders.")

# 2. Final Predict Function for Gradio
def predict_digit(input_img):
    img = Image.fromarray(input_img).convert('L')
    img = ImageOps.invert(img).resize((28, 28))
    tensor = transforms.ToTensor()(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(tensor)
        distances = torch.norm(prototypes - features, dim=1)
        pred_idx = torch.argmin(distances).item()
        # Returns something like "Sanskrit_character01"
        return class_names[pred_idx]
    

❌ Critical: No images found. Check if images_background contains alphabet folders.


Traceback (most recent call last):
  File "c:\Users\hec\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hec\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hec\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hec\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hec\AppData\Local\Programs\Py

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torchvision import transforms

# 1. Re-define Architecture (Must match your trained model)
class Conv4(nn.Module):
    def __init__(self):
        super(Conv4, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
    def forward(self, x):
        x = self.layer(x)
        return x.view(x.size(0), -1)

# 2. Setup Device and Load Model
device = torch.device("cpu")
model = Conv4().to(device)

# Load your saved weights (Make sure this filename matches exactly)
model.load_state_dict(torch.load('trained_fewshot_model.pth', map_location=device))
model.eval()

def predict_digit(input_img):
    # Process image: Gray -> Invert -> Resize to 28x28
    img = Image.fromarray(input_img).convert('L')
    img = ImageOps.invert(img) 
    img = img.resize((28, 28))
    
    # Convert to Tensor
    transform = transforms.Compose([transforms.ToTensor()])
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # 1. Get features for the uploaded image
        features = model(tensor)
        
        # 2. Calculate distance to all prototypes
        # (Assuming 'prototypes' is a tensor of your class averages)
        distances = torch.norm(prototypes - features, dim=1)
        
        # 3. Find the closest match
        pred_idx = torch.argmin(distances).item()
        
        # 4. Get the label (e.g., "22")
        predicted_label = class_names[pred_idx]
        
    return f"Predicted Digit: {predicted_label}"

# 3. Launch Interface
demo = gr.Interface(
    fn=predict_digit,
    inputs=gr.Image(),
    outputs="text",
    title="Handwriting Recognizer",
    description="Upload your notebook photo here!"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
